# Coastal Ecuador Biophysical Data Acquisition (2002-2025)

Downloads the five satellite/reanalysis variables used in the study for the
Ecuadorian coastal zone (1S-2S, west to 81.83W): sea surface temperature (SST),
sea surface salinity (SSS), mean sea level (MSL), 10-m wind, and chlorophyll-a.
Data are aggregated into weekly composites and saved as NetCDF files, one file
per variable per week, following the 1200-week schedule defined in
`weekly_schedule.xlsx` (see `data/README.md`).

**Sources**
- SST, SSS, MSL: Copernicus Marine Service (CMEMS), `GLOBAL_MULTIYEAR_PHY_001_030`
- Wind (10 m): ERA5 reanalysis, Copernicus Climate Data Store (CDS)
- Chlorophyll-a: NASA Ocean Color, MODIS-Aqua L3m daily, 4 km

**Credentials required** (none are stored in this notebook):
- A Copernicus Marine Service account (`copernicusmarine` login, prompted interactively)
- A NASA Earthdata account (`earthaccess` login, prompted interactively)
- A Copernicus Climate Data Store (CDS) personal access token, read from the
  `CDS_API_KEY` environment variable or your local `~/.cdsapirc` file — see
  Section 3 below.

**Structure**
- 0. Configuration (paths, study-area bounding box, weekly schedule)
- 1. Sea surface temperature, salinity, and mean sea level (CMEMS)
- 2. Chlorophyll-a (NASA Ocean Color)
- 3. Wind at 10 m (ERA5 / CDS)
- 4. Quality control of all downloaded files


## 0. Configuration

Paths, study-area bounding box, and the weekly time schedule shared by all variables.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import pandas as pd

# --- Root folder in Drive ---------------------------------------------------
# Edit this to point to your own working folder. All downloaded files and the
# weekly schedule spreadsheet are read from / written to subfolders of this path.
DATA_ROOT = "/content/drive/MyDrive/oceanographic_data/raw"

# Weekly schedule (start_date, end_date for each of the 1200 weeks, 2002-2025).
WEEKS_FILE = os.path.join(DATA_ROOT, "weekly_schedule.xlsx")

# --- Study-area bounding box -------------------------------------------------
MIN_LON, MAX_LON = -81.83, -80.67   # 81°50'W to 80°40'W
MIN_LAT, MAX_LAT = -2.00, -1.00     # 2S to 1S

weeks_df = pd.read_excel(WEEKS_FILE)
weeks_df.columns = ["start_date", "end_date"]
weeks_df["start_date"] = pd.to_datetime(weeks_df["start_date"])
weeks_df["end_date"] = pd.to_datetime(weeks_df["end_date"])

print(f"Weekly schedule loaded: {len(weeks_df)} weeks, "
      f"{weeks_df['start_date'].min().date()} to {weeks_df['end_date'].max().date()}")


## 1. Sea surface temperature, salinity, and mean sea level (CMEMS)

Downloads weekly composites of SST (`thetao`), SSS (`so`), and MSL (`zos`) from
the CMEMS global ocean physics reanalysis (`GLOBAL_MULTIYEAR_PHY_001_030`,
0.083 degree resolution). For the two 3D variables (SST, SSS), the surface
level (~0.49 m, the first available depth) is selected; MSL is natively 2D.
Each week is averaged from daily fields and saved as a separate NetCDF file
per variable.

In [ ]:
!pip install -q copernicusmarine xarray pandas openpyxl

import os
import numpy as np
import xarray as xr
import copernicusmarine as cm

cmems_root = os.path.join(DATA_ROOT, "CMEMS_weekly")
cmems_folders = {
    "SST": os.path.join(cmems_root, "SST"),
    "SSS": os.path.join(cmems_root, "SSS"),
    "MSL": os.path.join(cmems_root, "MSL"),
}
for path in cmems_folders.values():
    os.makedirs(path, exist_ok=True)

cmems_dataset_id = "cmems_mod_glo_phy_my_0.083deg_P1D-m"
cmems_variables = {
    "SST": "thetao",  # sea water potential temperature
    "SSS": "so",       # sea water salinity
    "MSL": "zos",      # sea surface height above geoid
}

# Surface level: request 0-1 m and select the nearest level to this depth.
target_depth = 0.494
min_depth, max_depth = 0.0, 1.0


In [ ]:
# Interactive login (prompts for Copernicus Marine Service credentials; not stored here)
cm.login()


In [ ]:
# Skip weeks that have already been downloaded for all three variables.
import re

existing_by_var = {}
for var, folder in cmems_folders.items():
    pattern = re.compile(rf"{var}_week_(\d{{4}})\.nc")
    found = {int(m.group(1)) for f in os.listdir(folder) if (m := pattern.match(f))}
    existing_by_var[var] = found

missing_weeks = [
    w for w in range(1, len(weeks_df) + 1)
    if not all(w in existing_by_var[var] for var in cmems_variables)
]
print(f"Weeks still pending: {len(missing_weeks)} of {len(weeks_df)}")


In [ ]:
# Set to an integer (e.g. 3) to test on a few weeks first; None downloads everything pending.
MAX_WEEKS = None
weeks_to_run = missing_weeks if MAX_WEEKS is None else missing_weeks[:MAX_WEEKS]

for week_index in weeks_to_run:
    row = weeks_df.iloc[week_index - 1]
    start, end = row["start_date"], row["end_date"]
    print(f"Week {week_index:04d}: {start.date()} to {end.date()}")

    ds = cm.open_dataset(
        dataset_id=cmems_dataset_id,
        variables=list(cmems_variables.values()),
        minimum_longitude=MIN_LON, maximum_longitude=MAX_LON,
        minimum_latitude=MIN_LAT, maximum_latitude=MAX_LAT,
        minimum_depth=min_depth, maximum_depth=max_depth,
        start_datetime=start, end_datetime=end,
        chunk_size_limit=-1,
    )

    ds_surface = ds.sel(depth=target_depth, method="nearest")
    weekly_mean = ds_surface.mean(dim="time", keep_attrs=True)

    time_coord = np.array([np.datetime64(start)])
    weekly_mean = weekly_mean.expand_dims(time=time_coord).assign_coords(time=time_coord)

    for var, var_name in cmems_variables.items():
        out_path = os.path.join(cmems_folders[var], f"{var}_week_{week_index:04d}.nc")
        if os.path.exists(out_path):
            continue
        weekly_mean[[var_name]].to_netcdf(out_path)
        print(f"  saved {out_path}")


## 2. Chlorophyll-a (NASA Ocean Color)

Downloads daily MODIS-Aqua L3m chlorophyll-a granules (4 km) for each week,
crops them to the study area, and averages them into a single weekly NetCDF
file. Weeks with no available granules are skipped and reported.

In [ ]:
!pip install -q earthaccess xarray pandas openpyxl

import os
import shutil
import numpy as np
import xarray as xr
import earthaccess

cloa_folder = os.path.join(DATA_ROOT, "CHL")
os.makedirs(cloa_folder, exist_ok=True)

cloa_short_name = "MODISA_L3m_CHL"
cloa_granule_pattern = "*.DAY.CHL.chlor_a.4km*"


In [ ]:
# Interactive login (prompts for NASA Earthdata credentials; not stored here)
auth = earthaccess.login(persist=True)


In [ ]:
# Set to an integer to test on a few weeks first; None processes the full schedule.
MAX_WEEKS = None

for idx, row in weeks_df.iterrows():
    week_index = idx + 1
    if MAX_WEEKS is not None and week_index > MAX_WEEKS:
        break

    start, end = row["start_date"], row["end_date"]
    out_path = os.path.join(cloa_folder, f"CHL_week_{week_index:04d}.nc")
    if os.path.exists(out_path):
        continue

    print(f"Week {week_index:04d}: {start.date()} to {end.date()}")

    tmp_dir = "/content/chl_tmp"
    if os.path.exists(tmp_dir):
        shutil.rmtree(tmp_dir)
    os.makedirs(tmp_dir, exist_ok=True)

    temporal = (start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
    try:
        results = earthaccess.search_data(
            short_name=cloa_short_name, temporal=temporal,
            granule_name=cloa_granule_pattern, count=500,
        )
    except Exception as e:
        print(f"  search failed ({e}), skipping week")
        continue

    if len(results) == 0:
        print("  no granules available for this week, skipping")
        continue

    try:
        paths = earthaccess.download(results, tmp_dir)
    except Exception as e:
        print(f"  download failed ({e}), skipping week")
        continue

    da_list = []
    for p in paths:
        try:
            with xr.open_dataset(p) as ds:
                chl = ds["chlor_a"].sel(
                    lat=slice(MAX_LAT, MIN_LAT),
                    lon=slice(MIN_LON, MAX_LON),
                )
                prod_name = ds.attrs.get("product_name", os.path.basename(p))
                try:
                    date_str = prod_name.split(".")[1][:8]
                    t = np.datetime64(date_str)
                except Exception:
                    t = np.datetime64(start.strftime("%Y-%m-%d"))
                da_list.append(chl.expand_dims(time=[t]))
        except Exception as e:
            print(f"  could not read {p}: {e}")

    if not da_list:
        print("  no readable granules, skipping week")
        continue

    chl_week = xr.concat(da_list, dim="time").mean(dim="time", keep_attrs=True)
    time_coord = np.array([np.datetime64(start.strftime("%Y-%m-%d"))])

    ds_week = xr.Dataset(
        {"chlor_a": chl_week},
        coords={"lat": chl_week["lat"], "lon": chl_week["lon"], "time": time_coord},
    )
    ds_week["chlor_a"].attrs.update({
        "long_name": "Weekly mean chlorophyll-a",
        "units": "mg m-3",
        "source": "MODIS-Aqua L3m DAY CHL chlor_a 4km",
        "note": "Weekly mean of daily L3m granules within the 7-day period.",
    })
    ds_week.to_netcdf(out_path)
    print(f"  saved {out_path}")


## 3. Wind at 10 m (ERA5 / CDS)

Downloads hourly ERA5 wind components (u10, v10), resamples them to daily and
then weekly means, and derives wind speed and direction. ERA5 is retrieved one
calendar month at a time to keep individual requests small.

**CDS credentials.** This notebook never stores your CDS API key as text. Set
it as an environment variable before running this section, either:

```python
import os
os.environ["CDS_API_KEY"] = "<your-personal-access-token>"
```

or, preferably, create a `~/.cdsapirc` file (the `cdsapi` client reads it
automatically and no key needs to be set in the notebook at all):

```
url: https://cds.climate.copernicus.eu/api
key: <your-personal-access-token>
```

A free token can be generated from your CDS account profile page after
accepting the ERA5 dataset license.

In [ ]:
!pip install -q cdsapi xarray netCDF4 pandas

import os
import calendar
import cdsapi
import xarray as xr
import numpy as np
import pandas as pd

wind_folder = os.path.join(DATA_ROOT, "WIND")
wind_dir_folder = os.path.join(DATA_ROOT, "WIND_DIRECTION")
era5_monthly_dir = os.path.join(DATA_ROOT, "ERA5_monthly")
for path in (wind_folder, wind_dir_folder, era5_monthly_dir):
    os.makedirs(path, exist_ok=True)

CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = os.environ.get("CDS_API_KEY")  # falls back to ~/.cdsapirc if unset

cds_client = cdsapi.Client(url=CDS_URL, key=CDS_KEY) if CDS_KEY else cdsapi.Client()


In [ ]:
# Download monthly ERA5 files for the full study period (skips months already on disk).
years = range(2002, 2026)

for year in years:
    for month in range(1, 13):
        out_nc = os.path.join(era5_monthly_dir, f"ERA5_wind_{year}_{month:02d}.nc")
        if os.path.exists(out_nc):
            continue

        ndays = calendar.monthrange(year, month)[1]
        request = {
            "product_type": "reanalysis",
            "variable": ["10m_u_component_of_wind", "10m_v_component_of_wind"],
            "year": str(year),
            "month": f"{month:02d}",
            "day": [f"{d:02d}" for d in range(1, ndays + 1)],
            "time": [f"{h:02d}:00" for h in range(24)],
            "area": [MAX_LAT, MIN_LON, MIN_LAT, MAX_LON],  # N, W, S, E
            "format": "netcdf",
        }
        print(f"Requesting ERA5 {year}-{month:02d}...")
        cds_client.retrieve("reanalysis-era5-single-levels", request, out_nc)


In [ ]:
# Combine monthly files into a single daily-mean dataset (kept in memory, not re-saved).
nc_files = sorted(
    os.path.join(era5_monthly_dir, f) for f in os.listdir(era5_monthly_dir) if f.endswith(".nc")
)
print(f"Monthly ERA5 files found: {len(nc_files)}")

ds_hourly = xr.open_mfdataset(nc_files, combine="by_coords")

rename_dict = {}
if "valid_time" in ds_hourly.dims:
    rename_dict["valid_time"] = "time"
if "latitude" in ds_hourly.dims:
    rename_dict["latitude"] = "lat"
if "longitude" in ds_hourly.dims:
    rename_dict["longitude"] = "lon"
ds_hourly = ds_hourly.rename(rename_dict)

u_var, v_var = "u10", "v10"
ds_daily = ds_hourly[[u_var, v_var]].resample(time="1D").mean(keep_attrs=True)
ds_daily = ds_daily.sel(time=slice(weeks_df["start_date"].min(), weeks_df["end_date"].max()))


In [ ]:
# Weekly wind speed and direction, derived from the daily u/v components.
for idx, row in weeks_df.iterrows():
    week_index = idx + 1
    start, end = row["start_date"], row["end_date"]

    speed_path = os.path.join(wind_folder, f"WIND_week_{week_index:04d}.nc")
    dir_path = os.path.join(wind_dir_folder, f"WIND_DIR_week_{week_index:04d}.nc")
    if os.path.exists(speed_path) and os.path.exists(dir_path):
        continue

    sub = ds_daily.sel(time=slice(start, end))
    if sub.time.size == 0:
        print(f"Week {week_index:04d}: no ERA5 daily data in this range, skipping")
        continue

    weekly_vec = sub[[u_var, v_var]].mean(dim="time", keep_attrs=True)

    if not os.path.exists(speed_path):
        speed = np.sqrt(weekly_vec[u_var] ** 2 + weekly_vec[v_var] ** 2)
        speed.name = "wind_speed_10m"
        speed_ds = weekly_vec.copy()
        speed_ds["wind_speed_10m"] = speed
        speed_ds.load().to_netcdf(speed_path, engine="h5netcdf")

    if not os.path.exists(dir_path):
        direction = (np.degrees(np.arctan2(-weekly_vec[u_var], -weekly_vec[v_var])) % 360)
        direction.name = "wind_dir_10m"
        dir_ds = xr.Dataset({"wind_dir_10m": direction}, coords=weekly_vec[u_var].coords)
        dir_ds.load().to_netcdf(dir_path, engine="h5netcdf")

    print(f"Week {week_index:04d}: wind speed and direction saved")


## 4. Quality control

Checks every expected weekly file against a plausible physical range for its
variable, and flags each week as `OK`, `POTENTIAL_ISSUE` (a meaningful fraction
of out-of-range values), or `MISSING`. The summary is written to
`quality_control.xlsx` in `DATA_ROOT`.

In [ ]:
!pip install -q tqdm

import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm

qc_folders = {
    "SST":       os.path.join(cmems_root, "SST"),
    "SSS":       os.path.join(cmems_root, "SSS"),
    "MSL":       os.path.join(cmems_root, "MSL"),
    "WIND":      wind_folder,
    "WIND_DIR":  wind_dir_folder,
    "CHL":       cloa_folder,
}

qc_var_names = {
    "SST": "thetao", "SSS": "so", "MSL": "zos",
    "WIND": "wind_speed_10m", "WIND_DIR": "wind_dir_10m", "CHL": "chlor_a",
}

# Plausible physical ranges, used only to flag clearly anomalous values.
qc_valid_ranges = {
    "SST": (0.0, 40.0),        # degrees C
    "SSS": (0.0, 45.0),        # PSU
    "MSL": (-5.0, 5.0),        # m
    "WIND": (0.0, 60.0),       # m/s
    "WIND_DIR": (0.0, 360.0),  # degrees
    "CHL": (0.0, 100.0),       # mg/m3
}

print("Folder check:")
for key, path in qc_folders.items():
    print(f"  {key:10s} {'found' if os.path.exists(path) else 'MISSING'}  {path}")


In [ ]:
def find_week_file(folder, key, week_str):
    """Locate the NetCDF file for a given week, tolerating minor naming variants."""
    candidate = os.path.join(folder, f"{key}_week_{week_str}.nc")
    if os.path.exists(candidate):
        return candidate
    matches = glob.glob(os.path.join(folder, f"*week_{week_str}*.nc"))
    return matches[0] if matches else None


def check_week_var(path, var, vmin, vmax, bad_frac_threshold=0.05):
    """Flags a file as OK, POTENTIAL_ISSUE, or MISSING based on the fraction
    of finite values that fall outside [vmin, vmax]."""
    if path is None or not os.path.exists(path):
        return "MISSING"
    try:
        with xr.open_dataset(path) as ds:
            if var not in ds:
                return "MISSING"
            vals = ds[var].values
    except Exception:
        return "MISSING"

    finite = np.isfinite(vals)
    if finite.sum() == 0:
        return "MISSING"

    data_vals = vals[finite]
    bad = (np.abs(data_vals) > 1e20) | (data_vals < vmin) | (data_vals > vmax)
    frac_bad = float(bad.sum()) / finite.sum()
    return "POTENTIAL_ISSUE" if frac_bad > bad_frac_threshold else "OK"


In [ ]:
records = []
for idx, row in tqdm(weeks_df.iterrows(), total=len(weeks_df), desc="Checking weeks", unit="week"):
    week_idx = idx + 1
    week_str = f"{week_idx:04d}"
    rec = {"week": week_idx, "start_date": row["start_date"].date(), "end_date": row["end_date"].date()}

    for key, folder in qc_folders.items():
        file_path = find_week_file(folder, key, week_str)
        rec[f"{key}_flag"] = check_week_var(
            file_path, qc_var_names[key], *qc_valid_ranges[key]
        )
    records.append(rec)

qc_df = pd.DataFrame(records)
qc_path = os.path.join(DATA_ROOT, "quality_control.xlsx")
qc_df.to_excel(qc_path, index=False)

print(f"Quality-control summary written to: {qc_path}")
qc_df.head()
